In [44]:
import os

for root, dirs, files in os.walk("."):
    for file in files:
        if file.endswith(".pth"):
            print(os.path.join(root, file))

./risk_model.pth


In [86]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class RiskPredictionModel(nn.Module):
    def __init__(self, feature_dim=6, hidden_dim=32, num_classes=3):
        super(RiskPredictionModel, self).__init__()

        self.fc1 = nn.Linear(feature_dim, hidden_dim)
        self.ln1 = nn.LayerNorm(hidden_dim)
        self.drop1 = nn.Dropout(0.3)

        self.fc2 = nn.Linear(hidden_dim, hidden_dim//2)
        self.ln2 = nn.LayerNorm(hidden_dim//2)
        self.drop2 = nn.Dropout(0.3)

        self.fc3 = nn.Linear(hidden_dim//2, num_classes)

    def forward(self, x):
        x = F.relu(self.ln1(self.fc1(x)))
        x = self.drop1(x)

        x = F.relu(self.ln2(self.fc2(x)))
        x = self.drop2(x)

        x = self.fc3(x)
        return x

In [46]:
def fuse_features(encoder_features, entropy_map, css_map):
    """
    encoder_features: [B, 12]
    entropy_map: [B, 1, H, W, D]
    css_map: [B, 1, H, W, D]
    """

    entropy_score = entropy_map.mean(dim=[2,3,4])
    css_score = css_map.mean(dim=[2,3,4])

    fused = torch.cat([encoder_features, entropy_score, css_score], dim=1)
    return fused

In [47]:
import torch

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

Using device: mps


In [ ]:
risk_model = RiskPredictionModel(feature_dim=6).to(device)

In [49]:
from monai.networks.nets import SwinUNETR

model = SwinUNETR(
    spatial_dims=3,
    in_channels=1,
    out_channels=3,
    feature_size=48
).to(device)

In [50]:
try:
    model.load_state_dict(torch.load("best_model.pth", map_location=device))
    print("Loaded saved model")
except:
    print("Using current trained model in memory")

model.eval()

Using current trained model in memory


SwinUNETR(
  (swinViT): SwinTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv3d(1, 48, kernel_size=(2, 2, 2), stride=(2, 2, 2))
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (layers1): ModuleList(
      (0): BasicLayer(
        (blocks): ModuleList(
          (0-1): 2 x SwinTransformerBlock(
            (norm1): LayerNorm((48,), eps=1e-05, elementwise_affine=True)
            (attn): WindowAttention(
              (qkv): Linear(in_features=48, out_features=144, bias=True)
              (attn_drop): Dropout(p=0.0, inplace=False)
              (proj): Linear(in_features=48, out_features=48, bias=True)
              (proj_drop): Dropout(p=0.0, inplace=False)
              (softmax): Softmax(dim=-1)
            )
            (drop_path): Identity()
            (norm2): LayerNorm((48,), eps=1e-05, elementwise_affine=True)
            (mlp): MLPBlock(
              (linear1): Linear(in_features=48, out_features=192, bias=True)
              (linear2): Linear(in_feature

In [51]:
import glob
import torch
from monai.transforms import Compose, LoadImaged, EnsureChannelFirstd, CenterSpatialCropd, ToTensord
from monai.data import Dataset, DataLoader

# Device
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# BraTS path
brats_path = "/Users/apple/Desktop/BRAINIAC/BraTS2020_ValidationData/MICCAI_BraTS2020_ValidationData"
cases = sorted(glob.glob(brats_path + "/*"))

# Build data list (FLAIR only)
data = []
for case in cases:
    flair_files = glob.glob(case + "/*_flair.nii")
    if len(flair_files) > 0:
        data.append({"image": flair_files[0]})

# Transforms (crop to 128)
transforms = Compose([
    LoadImaged(keys=["image"]),
    EnsureChannelFirstd(keys=["image"]),
    CenterSpatialCropd(keys=["image"], roi_size=(128,128,128)),
    ToTensord(keys=["image"])
])

# Loader
ds = Dataset(data=data, transform=transforms)
train_loader = DataLoader(ds, batch_size=1)

print("Loader ready:", len(data), "cases")

Loader ready: 125 cases


In [52]:
def compute_tumor_volume(seg_output):
    tumor = (seg_output.argmax(dim=1) > 0).float()
    volume = tumor.sum(dim=[1,2,3]) / (128*128*128)
    return volume.unsqueeze(1)

In [80]:
def get_risk_label(volume):
    if volume < 0.02:
        return 0   # Low Risk
    elif volume < 0.05:
        return 1   # Medium Risk
    else:
        return 2   # High Risk

In [53]:
for batch in train_loader:
    x = batch["image"].to(device)
    print("Input ready:", x.shape)
    break

Input ready: torch.Size([1, 1, 128, 128, 128])


In [70]:
import torch
import torch.nn.functional as F

# Entropy from feature maps
def compute_entropy_attention(features):
    probs = torch.softmax(features, dim=1)
    entropy = -torch.sum(probs * torch.log(probs + 1e-8), dim=1, keepdim=True)
    return entropy


# CSS map (feature deviation proxy)
def compute_css_map(model, x):
    with torch.no_grad():
        out = model(x)
    css = torch.std(out, dim=1, keepdim=True)
    return css


def fuse_features(encoder_features, entropy_map, css_map, seg_output):

    enc = encoder_features.mean(dim=[2,3,4])
    ent = entropy_map.mean(dim=[2,3,4])
    css = css_map.mean(dim=[2,3,4])
    vol = compute_tumor_volume(seg_output)

    fused = torch.cat([enc, ent, css, vol], dim=1)
    return fused

In [95]:
risk_model.train()

for i, batch in enumerate(train_loader):

    x = batch["image"].to(device)

    with torch.no_grad():
        seg_output = model(x)
        entropy_map = compute_entropy_attention(seg_output)
        css_map = compute_css_map(model, x)
        volume = compute_tumor_volume(seg_output)

    label = torch.tensor([get_risk_label(volume.item())]).to(device)

    fused_features = fuse_features(seg_output, entropy_map, css_map, seg_output)

    output = risk_model(fused_features)

    loss = criterion(output, label)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if i % 10 == 0:
        print("Step:", i, "Loss:", loss.item())

    if i == 50:   # train on 50 MRIs only (fast)
        break

Step: 0 Loss: 0.8607670068740845
Step: 10 Loss: 0.9297077655792236
Step: 20 Loss: 0.7416561245918274
Step: 30 Loss: 0.47047582268714905
Step: 40 Loss: 0.34782111644744873
Step: 50 Loss: 0.5827164649963379


In [96]:
risk_model = RiskPredictionModel(feature_dim=6).to(device)

In [97]:
optimizer = torch.optim.Adam(risk_model.parameters(), lr=1e-3)
criterion = torch.nn.CrossEntropyLoss()

risk_model.train()

for batch in train_loader:

    x = batch["image"].to(device)

    with torch.no_grad():
        seg_output = model(x)
        entropy_map = compute_entropy_attention(seg_output)
        css_map = compute_css_map(model, x)
        volume = compute_tumor_volume(seg_output)

    label = torch.tensor([get_risk_label(volume.item())]).to(device)

    fused_features = fuse_features(seg_output, entropy_map, css_map, seg_output)

    output = risk_model(fused_features)

    loss = criterion(output, label)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print("Training loss:", loss.item())
    break

Training loss: 1.9518859386444092


In [89]:
risk_model.eval()

with torch.no_grad():
    output = risk_model(fused_features)
    risk_probs = torch.softmax(output, dim=1)

risk_labels = ["Low Risk", "Medium Risk", "High Risk"]
pred_class = torch.argmax(risk_probs, dim=1)

print("Predicted Risk:", risk_labels[pred_class.item()])
print("Probabilities:", risk_probs.cpu().numpy())

Predicted Risk: Low Risk
Probabilities: [[0.4138402  0.21676381 0.369396  ]]


In [98]:
risk_labels = ["Low Risk", "Medium Risk", "High Risk"]
pred_class = torch.argmax(risk_probs, dim=1)

print("Predicted Risk:", risk_labels[pred_class.item()])
print("Probabilities:", risk_probs.cpu().numpy())

Predicted Risk: Low Risk
Probabilities: [[0.4138402  0.21676381 0.369396  ]]


In [99]:
#add the path of the models and save it inside that as risk_model.pth
torch.save(risk_model.state_dict(), "risk_model.pth")

In [100]:
# If you have already trained risk_model with feature_dim=6, re-save it:
torch.save(risk_model.state_dict(), "risk_model.pth")

# Now you can safely load it again without size mismatch
risk_model.load_state_dict(torch.load("risk_model.pth"))
risk_model.eval()

with torch.no_grad():
    output = risk_model(fused_features)
    temperature = 0.3
prob = torch.softmax(output / temperature, dim=1)

print("Final Risk Probability:", prob)

Final Risk Probability: metatensor([[0.2762, 0.6553, 0.0685]], device='mps:0')


In [101]:
risk_labels = ["Low Risk", "Medium Risk", "High Risk"]

probs = risk_probs.cpu().numpy()[0]
pred_class = probs.argmax()

print("Predicted Risk Level:", risk_labels[pred_class])
print("Confidence:", probs[pred_class])

Predicted Risk Level: Low Risk
Confidence: 0.4138402
